In [1]:
import os
import pandas as pd

# Directory containing the .flow files
directory = 'ISCX_TOR/scale_0.001/'

# List to store individual dataframes
dataframes = []

# Loop through each file in the directory
for filename in os.listdir(directory):
    if filename.endswith('.flow'):
        # Read the file into a dataframe
        df = pd.read_csv(os.path.join(directory, filename), low_memory=False)  # Adjust the reading method if necessary
        
        # Add a label column with the filename (without extension) as the label value
        df['label'] = os.path.splitext(filename)[0]
        
        # Append the dataframe to the list
        dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_df = pd.concat(dataframes, ignore_index=True)

# Display the combined dataframe
print(combined_df)


         TotBytes  SrcBytes  DstBytes  SrcGap  DstGap  sMeanPktSz  dMeanPktSz  \
0             597       597         0     0.0     0.0       597.0         0.0   
1              60         0        60     0.0     0.0         0.0        60.0   
2             657       597        60     0.0     0.0       597.0        60.0   
3             651        54       597     0.0     0.0        54.0       597.0   
4            1458        54      1404     0.0     0.0        54.0      1404.0   
...           ...       ...       ...     ...     ...         ...         ...   
2904306        54        54         0     0.0     0.0        54.0         0.0   
2904307       657       597        60     0.0     0.0       597.0        60.0   
2904308      1308       651       657     0.0     0.0       325.5       328.5   
2904309       597         0       597     0.0     0.0         0.0       597.0   
2904310       657       597        60     0.0     0.0       597.0        60.0   

         sMaxPktSz  dMaxPkt

In [2]:
import os
import yaml
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import re

def keep_features(df, features_to_keep):
    """
    Drop all columns from the DataFrame except for the specified features.
    
    Parameters:
    - df: pd.DataFrame, the input DataFrame
    - features_to_keep: list, list of column names to retain
    
    Returns:
    - pd.DataFrame with only the specified columns
    """
    # Ensure that the features_to_keep are in the DataFrame
    features_to_keep = [feature for feature in features_to_keep if feature in df.columns]
    
    # Return a DataFrame with only the specified features
    return df[features_to_keep]

# Function to preprocess each dataset
def preprocess_dataset(df):
    # Drop columns that contain only missing values
    df = df.dropna(axis=1, how='all')
    # Separate numeric and non-numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
    
    # Check if DataFrame has either numeric or non-numeric columns
    if not numeric_cols.empty or not non_numeric_cols.empty:
        # Handle missing values for numeric data
        if not numeric_cols.empty:
            imputer_numeric = SimpleImputer(strategy='mean')
            df_numeric = pd.DataFrame(imputer_numeric.fit_transform(df[numeric_cols]), columns=numeric_cols)
        else:
            df_numeric = pd.DataFrame()  # Empty DataFrame for numeric data if no numeric columns exist
        
        # Handle missing values for non-numeric data
        if not non_numeric_cols.empty:
            imputer_non_numeric = SimpleImputer(strategy='most_frequent')
            df_non_numeric = pd.DataFrame(imputer_non_numeric.fit_transform(df[non_numeric_cols]), columns=non_numeric_cols)
            # Convert non-numeric features to one-hot encoding
            encoder = OneHotEncoder(drop='first')
            encoded = encoder.fit_transform(df_non_numeric)
            encoded_df = pd.DataFrame(encoded.toarray(), columns=encoder.get_feature_names_out(non_numeric_cols))
        else:
            encoded_df = pd.DataFrame()  # Empty DataFrame for encoded non-numeric data if no non-numeric columns exist
        
        # Concatenate processed numeric and encoded non-numeric data
        df_preprocessed = pd.concat([df_numeric, encoded_df], axis=1)
        '''
        features = [
            'sTtl', 'Mean', 'SrcBytes', 'TotBytes', 'IdleTime', 'DstWin', 'SrcWin', 'sMeanPktSz', 'AckDat', 
            'TcpRtt', 'TcpOpt_MwsS  T', 'dTtl', 'SynAck', 'Rate', 'Min', 'Dur', 'pLoss', 'Flgs_ e s      ',
             'State_S_', 'PCRatio', 'TotPkts', 'DstPkts', 'dMeanPktSz'
        ]
        
        '''
        features = [
            'sTtl', 'AckDat', 'TcpRtt',
            'TcpOpt_MwsS  T','dTtl', 'SynAck',
            'State_S_', 'PCRatio'
        ]
        
        # Drop all columns except the ones in features_to_keep
        df_preprocessed = keep_features(df_preprocessed, features)  

        # Scale features
        scaler = MinMaxScaler()
        df_scaled = pd.DataFrame(scaler.fit_transform(df_preprocessed), columns=df_preprocessed.columns)

        return df_scaled
    else:
        return pd.DataFrame()
    
# Step 1: Separate the label column
X = combined_df.drop(columns=['label'])
y = combined_df['label']

pre_df = preprocess_dataset(X)

pre_df['label'] = y

print(len(pre_df.columns))
pre_df['label']

8


0          audio
1          audio
2          audio
3          audio
4          audio
           ...  
2904306     voip
2904307     voip
2904308     voip
2904309     voip
2904310     voip
Name: label, Length: 2904311, dtype: object

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, classification_report
import numpy as np

# Assuming `df` is your DataFrame and 'label' is your target column

# Step 1: Separate features (X) and target (y)
X = pre_df.drop(columns=['label'])  # Drop the label column to get the features
y = pre_df['label']  # Target variable

# Step 2: Initialize the Random Forest Classifier
clf = RandomForestClassifier(n_estimators=10, random_state=42)

# Step 3: Perform cross-validation
cv_results = cross_validate(clf, X, y, cv=5, scoring=['accuracy', 'f1_macro', 'recall_macro', 'precision_macro'], return_train_score=False)

# Print cross-validation results
print(f"Cross-Validation Accuracy: {np.mean(cv_results['test_accuracy']):.4f} ± {np.std(cv_results['test_accuracy']):.4f}")
print(f"Cross-Validation F1 Score: {np.mean(cv_results['test_f1_macro']):.4f} ± {np.std(cv_results['test_f1_macro']):.4f}")
print(f"Cross-Validation Recall: {np.mean(cv_results['test_recall_macro']):.4f} ± {np.std(cv_results['test_recall_macro']):.4f}")
print(f"Cross-Validation Precision: {np.mean(cv_results['test_precision_macro']):.4f} ± {np.std(cv_results['test_precision_macro']):.4f}")

# Optionally, fit the model on the entire dataset for feature importance
clf.fit(X, y)

# Step 4: Feature Importances
importances = clf.feature_importances_
indices = importances.argsort()[::-1]  # Sort in descending order

# Step 5: Print top 10 feature importances
print("\nTop 10 Feature Importances:")
for i in range(min(10, len(indices))):  # Ensure there are at least 10 features
    print(f"Feature {X.columns[indices[i]]}: {importances[indices[i]]:.4f}")

# If you still want to compute confusion matrix and classification report, you need to split data again and train/test model
# For demonstration purposes, let's use the initial train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Confusion Matrix and Classification Report
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


d:\Dataset\IUST_MSCN_Dataset\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Dataset\IUST_MSCN_Dataset\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Dataset\IUST_MSCN_Dataset\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Dataset

Cross-Validation Accuracy: 0.3150 ± 0.0299
Cross-Validation F1 Score: 0.0934 ± 0.0361
Cross-Validation Recall: 0.1509 ± 0.0279
Cross-Validation Precision: 0.2286 ± 0.1111

Top 10 Feature Importances:
Feature dTtl: 0.7936
Feature AckDat: 0.0812
Feature SynAck: 0.0700
Feature TcpRtt: 0.0531
Feature sTtl: 0.0017
Feature State_S_: 0.0004
Feature PCRatio: 0.0000

Confusion Matrix:
[[  1181      6      0      0      0      0      1  13663]
 [     0   1373      0      0      0      0    873  35078]
 [     0      5      0      0      0      0      0   2717]
 [     0      3      0     29      0      0      0 197360]
 [     0      0      0      0      0      0      0  19314]
 [     0      1      0      0      0     61      0 223109]
 [     0      9      0      0      0      0  13319 102636]
 [     0      1      0      0      0      0      0 260555]]

Classification Report:


d:\Dataset\IUST_MSCN_Dataset\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Dataset\IUST_MSCN_Dataset\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

       audio       1.00      0.08      0.15     14851
    browsing       0.98      0.04      0.07     37324
        chat       0.00      0.00      0.00      2722
         ftp       1.00      0.00      0.00    197392
        mail       0.00      0.00      0.00     19314
         p2p       1.00      0.00      0.00    223171
       video       0.94      0.11      0.20    115964
        voip       0.30      1.00      0.47    260556

    accuracy                           0.32    871294
   macro avg       0.65      0.15      0.11    871294
weighted avg       0.76      0.32      0.17    871294



d:\Dataset\IUST_MSCN_Dataset\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report, confusion_matrix, precision_score
from sklearn.utils.class_weight import compute_class_weight

# Read the DataFrame (Replace this with your DataFrame reading code)
df = pre_df

# List of features to compute incremental statistics
feature_list = ['SynAck', 'TcpRtt', 'AckDat']  # Replace with your actual features

import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

def analyze_features(df):
    """
    Analyzes features of the DataFrame based on statistical properties and diversity.
    
    Parameters:
    - df: pandas DataFrame containing the features to analyze.
    
    Returns:
    - stats_df_sorted: pandas DataFrame containing the statistical metrics for each feature, sorted by variance.
    """
    # Separate features (assuming the last column is the target variable)
    X = df.drop(columns=[df.columns[-1]])  # Drop the last column assuming it's the target
    
    # Initialize a DataFrame to hold statistical metrics
    stats_df = pd.DataFrame(index=X.columns)
    
    # Calculate statistical metrics
    stats_df['Mean'] = X.mean()
    stats_df['Median'] = X.median()
    stats_df['StdDev'] = X.std()
    stats_df['Variance'] = X.var()
    stats_df['Range'] = X.max() - X.min()
    stats_df['Skewness'] = X.apply(lambda x: skew(x.dropna()))
    stats_df['Kurtosis'] = X.apply(lambda x: kurtosis(x.dropna()))
    stats_df['Missing Values'] = X.isna().sum()
    
    # Sort by variance (or any other metric you prefer)
    stats_df_sorted = stats_df.sort_values(by='Variance', ascending=False)
    
    # Print the sorted statistics
    print("Feature Statistical Analysis and Diversity:")
    print(stats_df_sorted)
    
    return stats_df_sorted

# Example usage:
# df = pd.read_csv('your_data.csv')  # Load your DataFrame
# stats_df_sorted = analyze_features(df)


# Compute incremental statistics
def compute_incremental_stats(df, features):
    incremental_stats = pd.DataFrame(index=df.index)
    
    for feature in features:
        incremental_stats[f'{feature}_mean'] = df[feature].expanding().mean()
        incremental_stats[f'{feature}_median'] = df[feature].expanding().median()
        incremental_stats[f'{feature}_std'] = df[feature].expanding().std()
        #incremental_stats[f'{feature}_max'] = df[feature].expanding().max()
    
    return incremental_stats

# Compute incremental statistics for the features
incremental_stats = compute_incremental_stats(df, feature_list)

# Fill NaN values in incremental statistics with 0
incremental_stats.fillna(0, inplace=True)

# Add incremental statistics to the original DataFrame
df_with_stats = pd.concat([df, incremental_stats], axis=1)

# Standardize the incremental features before clustering
incremental_feature_cols = [col for col in df_with_stats.columns if col.endswith(('mean', 'median', 'std', 'max'))]

X = df_with_stats[incremental_feature_cols]

analyze_features(X)

# Fill NaN values in the features (if any) with 0
X.fillna(0, inplace=True)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Perform MiniBatch KMeans clustering
n_clusters = 3
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels to the DataFrame
df_with_stats['cluster'] = clusters

# Remove incremental features
df_final = df_with_stats.drop(columns=incremental_feature_cols)

# Initialize dictionary to store classifiers and results
classifiers = {}
results = {}

# Train and evaluate Random Forest Classifier for each cluster with cross-validation
for i in range(n_clusters):
    # Select data for the current cluster
    cluster_data = df_final[df_final['cluster'] == i]
    
    if len(cluster_data) == 0:
        print(f"Cluster {i} has no data. Skipping.")
        continue
    
    # Separate features and target
    X_cluster = cluster_data.drop(columns=['label'])
    y_cluster = cluster_data['label']
    
    # Check if there is enough data to perform cross-validation
    if len(X_cluster) < 5:
        print(f"Not enough data for cross-validation in cluster {i}.")
        continue
    
    # Initialize Random Forest Classifier
    clf = RandomForestClassifier(n_estimators=10, random_state=42)
    
    # Perform cross-validation
    cv_results = cross_validate(clf, X_cluster, y_cluster, cv=5, scoring=['accuracy', 'f1_macro', 'recall_macro', 'precision_macro'], return_train_score=False)
    
    # Fit the model on the entire cluster data for confusion matrix
    clf.fit(X_cluster, y_cluster)
    
    # Predict on the same data to compute confusion matrix
    y_pred = clf.predict(X_cluster)
    
    # Compute confusion matrix
    cm = confusion_matrix(y_cluster, y_pred)
    
    # Store the classifier and results
    classifiers[i] = clf
    results[i] = {
        'cv_accuracy': np.mean(cv_results['test_accuracy']),
        'cv_f1_score': np.mean(cv_results['test_f1_macro']),
        'cv_recall': np.mean(cv_results['test_recall_macro']),
        'cv_precision': np.mean(cv_results['test_precision_macro']),
        'confusion_matrix': cm,
        'classification_report': classification_report(y_cluster, y_pred)
    }

# Print the results for each cluster
for i, result in results.items():
    print(f"\nCluster {i} - Random Forest Classifier Performance:")
    print(f"Cross-Validation Accuracy: {result['cv_accuracy']:.4f}")
    print(f"Cross-Validation F1 Score: {result['cv_f1_score']:.4f}")
    print(f"Cross-Validation Recall: {result['cv_recall']:.4f}")
    print(f"Cross-Validation Precision: {result['cv_precision']:.4f}")
    print("\nConfusion Matrix:")
    print(result['confusion_matrix'])
    print("\nClassification Report:")
    print(result['classification_report'])


Feature Statistical Analysis and Diversity:
                   Mean    Median    StdDev  Variance     Range  Skewness  \
TcpRtt_std     0.077743  0.064592  0.035424  0.001255  0.244691  1.576664   
SynAck_std     0.077719  0.064570  0.035416  0.001254  0.244660  1.577102   
TcpRtt_mean    0.008930  0.005005  0.009788  0.000096  0.079424  2.879136   
SynAck_mean    0.008927  0.005003  0.009786  0.000096  0.079415  2.879621   
AckDat_mean    0.002362  0.001420  0.002249  0.000005  0.014429  2.200661   
SynAck_median  0.000000  0.000000  0.000000  0.000000  0.000000       NaN   
TcpRtt_median  0.000000  0.000000  0.000000  0.000000  0.000000       NaN   
AckDat_median  0.000000  0.000000  0.000000  0.000000  0.000000       NaN   

                Kurtosis  Missing Values  
TcpRtt_std      2.694056               0  
SynAck_std      2.696130               0  
TcpRtt_mean    10.421406               0  
SynAck_mean    10.425561               0  
AckDat_mean     4.928071               0  
SynA

C:\Users\mjf\AppData\Local\Temp\ipykernel_18076\1320444306.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.fillna(0, inplace=True)
d:\Dataset\IUST_MSCN_Dataset\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Dataset\IUST_MSCN_Dataset\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Dataset\IUST


Cluster 0 - Random Forest Classifier Performance:
Cross-Validation Accuracy: 0.4298
Cross-Validation F1 Score: 0.1231
Cross-Validation Recall: 0.1860
Cross-Validation Precision: 0.2029

Confusion Matrix:
[[   387      0      0      0      1  39218]
 [     0      0      0      0      0  14579]
 [     0      0      0      0      0  63855]
 [     0      0      0    222      1 745015]
 [    15      0      0      0  44729 343280]
 [     0      0      0      0      1 864677]]

Classification Report:
              precision    recall  f1-score   support

       audio       0.96      0.01      0.02     39606
         ftp       0.00      0.00      0.00     14579
        mail       0.00      0.00      0.00     63855
         p2p       1.00      0.00      0.00    745238
       video       1.00      0.12      0.21    388024
        voip       0.42      1.00      0.59    864678

    accuracy                           0.43   2115980
   macro avg       0.56      0.19      0.14   2115980
weighted avg